In [1]:
#Assorted imports
import numpy as np
import pandas as pd
import h5py
import vaex
import pynbody
from pynbody.array import SimArray

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from matplotlib.path import Path

from astropy import units as u
from astropy.io import ascii, fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord,CartesianRepresentation,match_coordinates_sky

import functions_altered as fn 
from matched_filter_ani_altered import matched_filter_ani as mf
import os

np.random.seed(0)

In [2]:
s = pynbody.load('/home/christenc/Storage/Cosmo/storm.cosmo25cmb/storm.cosmo25cmb.4096g5HbwK1BH/storm.cosmo25cmb.4096g5HbwK1BH.004096/storm.cosmo25cmb.4096g5HbwK1BH.004096')

h = s.halos(halo_numbers='v1')  # Load the halos using the original AHF numbering system
h.load_all()
unique_halo_ids = list(h.keys())


In [3]:
num = 2 
main_halo = h[num] #h[unique_halo_ids[0]]
#iterations = len(unique_halo_ids) - 1
#for i in range(1, iterations):
#    if len(main_halo) < len(h[unique_halo_ids[i]]):
#        main_halo = h[unique_halo_ids[i]]
#
# unique_halo_ids[0]

In [4]:
pynbody.analysis.halo.center
cen = main_halo.mean_by_mass('pos')

sp = s[pynbody.filt.Sphere(SimArray([200], "kpc"), cen)].load_copy()

s.physical_units()


with h5py.File('/home/christenc/Storage/Cosmo/storm.cosmo25cmb/storm.cosmo25cmb.4096g5HbwK1BH/storm.cosmo25cmb.4096g5HbwK1BH_allhalostardata_upd.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] 
    partids_h5 = f['particle_IDs'][:]

partids_snap = sp.s['iord']

In [5]:
box = 'storm' 
D = 2000 
mlim_str = '26p5' 
name = f'{box}_4096_{num}_data_{D}_{mlim_str}'

dwarfcatpath = f"/home/otteleno/MAP/matched_filter_starters/{box}_{num}_data/survey.{name}.0.h5" 
vdwarfcat = vaex.open(dwarfcatpath)
dwarfcat = pd.DataFrame(vdwarfcat,columns=vdwarfcat.column_names)

size_kpc = 40 
pdist = (size_kpc/D) * (180/np.pi) 
year = 10
mlim = '25'
c1='px'
c2='py'
edgelength = 10 
plotdir = f'{box}_4096_{num}' 

In [6]:
dwarfcat

,age,dec,dmod,feh,glat,glon,grav,lsst_g,lsst_g_Err,lsst_g_Intrinsic,...,py,pz,ra,rad,smass,teff,vr,vx,vy,vz
0,5.790941,-27.289413,26.505910,-3.353632,-89.838794,-58.435505,4.630040,24.973608,0.0,-1.532301,...,-4.796306,-2000.692459,12.863807,2000.700378,8.469514,30391.779297,22.424863,-15.012797,-118.182710,-22.163740
1,5.790941,-27.288862,26.505966,-3.353632,-89.839341,-58.489424,4.641106,25.128321,0.0,-1.377645,...,-4.782901,-2000.744140,12.863962,2000.752006,7.995754,29654.767578,22.711874,-14.970212,-117.942916,-22.451953
2,5.790941,-27.286993,26.505830,-3.353632,-89.841203,-58.591757,4.678921,25.665352,0.0,-0.840478,...,-4.732338,-2000.619082,12.864229,2000.626766,6.477076,26965.000000,22.309659,-14.936937,-118.317029,-22.051448
3,5.790941,-27.288070,26.505883,-3.353632,-89.840177,-57.554448,4.704493,26.048357,0.0,-0.457527,...,-4.709607,-2000.667868,12.861004,2000.675652,5.550234,25086.126953,22.424281,-14.807811,-118.348043,-22.167936
4,5.790941,-27.289710,26.505906,-3.353632,-89.838500,-58.381519,4.394523,22.729784,0.0,-3.776121,...,-4.802262,-2000.688145,12.863643,2000.696092,21.333363,42878.335938,22.726916,-15.012905,-118.214109,-22.465441
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276384,10.072595,-26.779000,26.495388,-1.231300,-89.650697,121.950746,2.083633,27.048178,0.0,0.552790,...,10.299319,-1990.991771,12.852778,1991.028771,0.841396,4841.669922,-14.144604,-50.328127,-165.952707,13.448771
276385,10.072689,-26.743475,26.495769,-1.238766,-89.612804,116.513608,0.443303,25.110546,-0.0,-1.385223,...,12.042001,-1991.333299,12.811010,1991.378770,0.842181,3946.756104,-14.839596,-49.814425,-161.079507,14.016135
276386,10.073256,-26.960854,26.497580,-1.077599,-89.832120,127.274086,1.573500,26.327957,-0.0,-0.169623,...,4.646928,-1993.031053,12.873738,1993.039608,0.850544,4515.127441,-13.319804,-44.434561,-195.750620,12.942301
276387,10.073727,-27.040387,26.496199,-1.075377,-89.801335,-173.271817,1.724962,26.549034,-0.0,0.052835,...,-0.809125,-1991.761102,13.059603,1991.773075,0.850442,4598.263184,17.371433,-33.724327,-193.655836,-17.176738


In [7]:
ananke_data = np.column_stack([dwarfcat['px'],dwarfcat['py'], dwarfcat['pz']])
ananke_masses = dwarfcat['smass']
ananke_total_mass = np.sum(ananke_masses)

cm_current_ananke = [0,0,0]
for i in range(len(ananke_data)):
    cm_current_ananke = cm_current_ananke + ananke_data[i] * ananke_masses[i]
cm_ananke = [x/ananke_total_mass for x in cm_current_ananke]


ananke_data_centered = ananke_data - cm_ananke
print(ananke_data_centered)

[[ 2.64682261 -4.84137451 -0.9452555 ]
 [ 2.63239124 -4.82796989 -0.99693681]
 [ 2.5897773  -4.777407   -0.87187901]
 ...
 [-3.83647927  4.60185946  6.7161503 ]
 [-7.15842088 -0.85419338  7.98610165]
 [-7.47069343 -3.78152763  6.63070156]]


In [5]:
def fibonacci_angler(n):

    golden_ratio = (1 + np.sqrt(5))/2

    i_array = np.linspace(0,n-1,n)
    z_array_double = 1 - i_array/((n-1)) ###ONLY GOES HALF WAY DOWN
    radius_array = np.sqrt(1-z_array**2)

    declination_array = np.pi/2-np.arcsin(z_array)
    azimuthal_array = 2*np.pi * i_array/golden_ratio
    

    return declination_array, azimuthal_array

In [6]:
declination_array, azimuthal_array = fibonacci_angler(24)

NameError: name 'z_array' is not defined

In [13]:
def new_axes(host_ids):

    #Calculates statistics for the entire galaxy
    
    m = sp.s['mass']
    x = sp.s['pos'][:, 0]
    y = sp.s['pos'][:, 1]
    z = sp.s['pos'][:, 2]
    vx = sp.s['vel'][:, 0]
    vy = sp.s['vel'][:, 1]
    vz = sp.s['vel'][:, 2]

    particle_array_all = np.column_stack((m,x,y,z,vx,vy,vz))
    pos_array_all = particle_array_all[:, 1:4]
    vel_array_all = particle_array_all[:, 4:7]
    
    total_mass_all=np.sum(m)
    
    cm_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        cm_current_all = cm_current_all + particle_array_all[i][0] * pos_array_all[i]
    cm_all = cm_current_all/total_mass_all
    
    avg_vel_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        avg_vel_current_all = avg_vel_current_all + particle_array_all[i][0] * vel_array_all[i]
    avg_vel_all = avg_vel_current_all/total_mass_all
    
    pos_array_centered_all = pos_array_all - cm_all
    vel_array_centered_all = vel_array_all - avg_vel_all


    #Calculates statistics only for the halo of interest, but adjusts them using the entire galaxy
    
    _, idloc_snap, idloc_h5 = np.intersect1d(partids_snap, partids_h5, return_indices = True) 
                                                                                             
    
    progenitor = hostids[idloc_h5] 
    mask = np.isin(progenitor, host_ids) 
    
    m_halo = m[idloc_snap][mask] 
    x_halo = x[idloc_snap][mask]
    y_halo = y[idloc_snap][mask]
    z_halo = z[idloc_snap][mask]
    vx_halo = vx[idloc_snap][mask]
    vy_halo = vy[idloc_snap][mask]
    vz_halo = vz[idloc_snap][mask]

    particle_array_halo = np.column_stack((m_halo,x_halo,y_halo,z_halo,vx_halo,vy_halo,vz_halo)) 
    pos_array_halo = particle_array_halo[:, 1:4]
    vel_array_halo = particle_array_halo[:, 4:7]
    pos_array_centered_halo = pos_array_halo - cm_all 
    vel_array_centered_halo = vel_array_halo - avg_vel_all 
    
    total_mass_halo=np.sum(m_halo)

    cm_current_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        cm_current_halo = cm_current_halo + particle_array_halo[i][0] * pos_array_centered_halo[i]
    cm_halo = [x/total_mass_halo for x in cm_current_halo]
    
    ang_mom_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        ang_mom_halo = ang_mom_halo + particle_array_halo[i][0] * np.cross(pos_array_centered_halo[i], vel_array_centered_halo[i])

    #Creates new basis and transforms data

    z_prime = ang_mom_halo/np.linalg.norm(ang_mom_halo)
    y_prime_unnormed = np.cross(ang_mom_halo, cm_halo)
    y_prime =  y_prime_unnormed/np.linalg.norm(y_prime_unnormed)
    x_prime = np.cross(z_prime, y_prime)
    transform_matrix = np.row_stack((x_prime, y_prime, z_prime)) 
    
    return transform_matrix
    
    

In [14]:
transform_matrix = new_axes('1344_29')

In [15]:
print(transform_matrix)

[[ 0.08107743 -0.99610972  0.03452346]
 [-0.57178798 -0.01811338  0.82020144]
 [ 0.8163853   0.08623992  0.57103216]]


In [16]:
def angle_view(coordinate_transform, declination, azimuthal,save = False, plot=True):

    '''Calculates 2d image coordinates based on angle of view. Plots graph. Many optional parameters for moviemaking ease

    host_ids = hostid(s) of halo to calibrate axes around, as well as color differently
    declination = camera angle measured down from z axis
    azimuthal = camera angle measured counterclockwise from x axis in xy plane
    
    Below are largely movie features 
    
    plot = Whether to display plot on Jupyter or not.
    save = Whether to save plot to file system
    axis_of_rotation = x,y, or z. Axis camera is rotation around.
    angle = Angle from current camera position to default camera position (depends on what type of movie you're making
    limits = 1x4 array of [min image(x), max image(x), min image(y), max image(y)]
    n = index of what image. Stored like 000 for first image, and 010 for 11th image for example.
    
    '''
    #Transforms data into new axes
    new_pos_array_all = np.matmul(transform_matrix, ananke_data_centered.T)

    #Spherical Coordinates
    theta = declination
    phi = azimuthal 
    declination_readable = round(declination*180/np.pi,1)
    azimuthal_readable = round(np.mod(azimuthal*180/np.pi, 360),1)

    #Uses generalized matrix to find 2d projected image from any given camera angle 
    project_matrix = np.array([[-np.sin(phi),                np.cos(phi),             0            ],
                             [-np.cos(theta)*np.cos(phi), -np.cos(theta)*np.sin(phi), np.sin(theta)]])

    project_pos_array_all = np.matmul(project_matrix, new_pos_array_all) #applies 2x3 transformation to 3xn data. Result is 2xn data 

    if plot == True:
        fig,ax  = plt.subplots()
        ax.scatter (project_pos_array_all[0], project_pos_array_all[1], s=1)
        ax.set_title(f"Declination = {declination_readable} degrees and Azimuthal = {azimuthal_readable} degrees")
        ax.set_title
        plt.show()
       

    return declination_readable, azimuthal_readable, project_pos_array_all



In [22]:
for i in range(len(declination_array)):
    d, a, altered_positions=  angle_view(transform_matrix, declination_array[i], azimuthal_array[i], plot=False)
    dwarfcat['px'] = altered_positions.T[:,0]
    dwarfcat['py'] = altered_positions.T[:,1]

    file_path = f"/home/otteleno/MAP/automated_data/rotated_catalogs/storm_2/{box}_{num}_d={d}_a={a}.h5"
    if os.path.exists(file_path):
        os.remove(file_path)

    vframe_new = vaex.from_pandas(dwarfcat)
    vframe_new.export_hdf5(file_path, progress=True)
    






export(hdf5) [########################################] 100.00% elapsed time  :     2.62s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.59s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.61s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.60s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.59s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.59s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.76s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.64s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.56s =  0.0m =  0.0h
export(hdf5) [################################

In [18]:
print(len(ananke_data))
print(len(sp))

276389
17074495
